# L24 — Goodness-of-Fit Testing

**Module**: M07 | **Chapter**: 9 | **Lecture**: L24

## Learning Objectives
By the end of this notebook you will be able to:
1. Apply the Kolmogorov-Smirnov test and interpret the p-value correctly.
2. Construct an equal-probability chi-squared test with the right degrees of freedom.
3. Produce and interpret Q-Q plots for non-Normal distributions.
4. Explain why a failed GoF test does not always mean reject-and-stop.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

Continuing with the clinic datasets from L22.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
data = pd.read_csv("../../../data/clinic_arrivals.csv")["interarrival_time"].to_numpy()
n = len(data)

# Best-fit parameters from L22 (recompute here for self-containment)
_, scale_exp = stats.expon.fit(data, floc=0)
a_gam, _, scale_gam = stats.gamma.fit(data, floc=0)
s_ln, _, scale_ln = stats.lognorm.fit(data, floc=0)

print(f"n = {n}")
print(f"Exponential: scale={scale_exp:.3f}")
print(f"Gamma:       shape={a_gam:.3f}, scale={scale_gam:.3f}")
print(f"Lognormal:   sigma={s_ln:.3f}, scale={scale_ln:.3f}")

## 1. Kolmogorov-Smirnov Test

The KS test measures the maximum vertical distance between the ECDF and the fitted CDF:
$$D_n = \sup_x |F_n(x) - F(x; \hat{\theta})|$$

**Important caveat**: when parameters are estimated from the same data,
the standard KS p-value is anti-conservative (too small).
Strictly, you should use the Lilliefors correction or a parametric bootstrap.
For a first pass, the uncorrected KS is common practice.

In [ ]:
alpha = 0.05

candidates = {
    'Exponential': stats.expon(loc=0, scale=scale_exp),
    'Gamma':       stats.gamma(a=a_gam, loc=0, scale=scale_gam),
    'Lognormal':   stats.lognorm(s=s_ln, loc=0, scale=scale_ln),
}

print(f"KS test results (α={alpha}):")
print(f"{'Distribution':15s}  {'D_n':8s}  {'p-value':10s}  {'Decision':20s}")
print('-' * 60)
for name, dist in candidates.items():
    ks_stat, p_val = stats.kstest(data, dist.cdf)
    decision = 'Fail to reject H₀' if p_val > alpha else 'Reject H₀'
    print(f"{name:15s}  {ks_stat:8.4f}  {p_val:10.4f}  {decision}")

## 2. Visualise the KS Statistic

The KS statistic is a geometric quantity — it's the largest gap between ECDF and CDF.

In [ ]:
x_sorted = np.sort(data)
ecdf_vals = np.arange(1, n+1) / n

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, dist) in zip(axes, candidates.items()):
    fitted_cdf = dist.cdf(x_sorted)
    gaps = np.abs(ecdf_vals - fitted_cdf)
    max_idx = np.argmax(gaps)

    ax.step(x_sorted, ecdf_vals, where='post', color='steelblue', lw=1.5, label='ECDF')
    x_plot = np.linspace(0, x_sorted.max(), 400)
    ax.plot(x_plot, dist.cdf(x_plot), color='tomato', lw=2, label=f'{name} CDF')

    # Mark the maximum gap
    x_gap = x_sorted[max_idx]
    y_ecdf = ecdf_vals[max_idx]
    y_cdf  = fitted_cdf[max_idx]
    ax.annotate('', xy=(x_gap, y_cdf), xytext=(x_gap, y_ecdf),
                arrowprops=dict(arrowstyle='<->', color='black', lw=2))
    ax.text(x_gap * 1.05, (y_ecdf + y_cdf) / 2,
            f'$D_n$={gaps[max_idx]:.3f}', fontsize=9)

    ks_stat, p_val = stats.kstest(data, dist.cdf)
    ax.set_title(f'{name}\np={p_val:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('F(x)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('KS test — maximum gap between ECDF and fitted CDF', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Chi-Squared Goodness-of-Fit Test

Equal-probability bins guarantee each bin has the same expected count,
which satisfies the requirement that expected count ≥ 5 per bin.

Degrees of freedom = (number of bins) − 1 − (number of estimated parameters)

In [ ]:
def chi2_gof(data, dist, n_params, n_bins=20, alpha=0.05):
    """Equal-probability chi-squared GoF test."""
    n = len(data)
    # Bin edges from quantiles of fitted distribution
    probs = np.linspace(0, 1, n_bins + 1)
    edges = dist.ppf(probs)
    edges[0]  = 0
    edges[-1] = np.inf

    observed, _ = np.histogram(data, bins=edges)
    expected = np.full(n_bins, n / n_bins, dtype=float)

    chi2_stat = ((observed - expected) ** 2 / expected).sum()
    df = n_bins - 1 - n_params
    p_val = 1 - stats.chi2.cdf(chi2_stat, df=df)
    decision = 'Fail to reject H₀' if p_val > alpha else 'Reject H₀'

    return chi2_stat, df, p_val, decision, observed, expected


n_params_map = {'Exponential': 1, 'Gamma': 2, 'Lognormal': 2}

print(f"Chi-squared GoF test (20 equal-prob bins, α={alpha}):")
print(f"{'Distribution':15s}  {'χ²':8s}  {'df':4s}  {'p-value':10s}  {'Decision'}")
print('-' * 65)
for name, dist in candidates.items():
    chi2_stat, df, p_val, decision, obs, exp = chi2_gof(
        data, dist, n_params_map[name]
    )
    print(f"{name:15s}  {chi2_stat:8.3f}  {df:4d}  {p_val:10.4f}  {decision}")

## 4. Q-Q Plot

A Q-Q plot compares the empirical quantiles to the theoretical quantiles.
Points that fall on the 45° line indicate a good fit.
Systematic bowing above the line indicates lighter tails than the fitted distribution;
bowing below indicates heavier tails.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plotting positions (Hazen formula)
p_plot = (np.arange(1, n+1) - 0.5) / n

for ax, (name, dist) in zip(axes, candidates.items()):
    theoretical_q = dist.ppf(p_plot)
    empirical_q   = x_sorted

    ax.scatter(theoretical_q, empirical_q, s=4, alpha=0.4, color='steelblue')

    # 45-degree reference line
    q_max = min(theoretical_q.max(), empirical_q.max())
    ax.plot([0, q_max], [0, q_max], 'r--', lw=1.5, label='45° line')

    ax.set_xlabel(f'{name} theoretical quantile')
    ax.set_ylabel('Empirical quantile')
    ax.set_title(f'Q-Q plot: {name}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Q-Q plots for clinic interarrival times', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Putting It Together: Distribution Selection

A structured selection decision uses all four pieces of evidence:

In [ ]:
# Collect evidence
evidence = {}
for name, dist in candidates.items():
    ks_stat, ks_p = stats.kstest(data, dist.cdf)
    chi2_stat, df, chi2_p, _, _, _ = chi2_gof(data, dist, n_params_map[name])
    ll = dist.logpdf(data).sum()
    k  = n_params_map[name]
    aic = -2 * ll + 2 * k
    evidence[name] = {
        'KS D_n': round(ks_stat, 4),
        'KS p':   round(ks_p, 4),
        'χ² stat': round(chi2_stat, 3),
        'χ² p':   round(chi2_p, 4),
        'AIC':    round(aic, 2),
    }

print(pd.DataFrame(evidence).T.to_string())

print()
print("Decision framework:")
print("  1. Which distributions fail-to-reject at α=0.05? (KS p > 0.05 AND χ² p > 0.05)")
print("  2. Among those, which has the lowest AIC?")
print("  3. Does the Q-Q plot show systematic deviations in the upper tail?")
print("  4. Which distribution is most defensible to domain experts?")

## 6. Sample Size Effect on Test Power

With larger n, even tiny distributional errors get detected.
This is not a bug — it means GoF tests become more demanding as you collect more data.

In [ ]:
# Run KS test at varying sample sizes, always against Exponential
rng = np.random.default_rng(42)
# Simulate data from the true Exponential with scale=12
true_scale = 12.0

sample_sizes = [50, 100, 200, 500, 1000, 5000]
n_trials = 500

print("Fraction of KS tests that reject Exponential (data IS Exponential):")
print("(Should be ~α=0.05 — this is the Type I error rate)")
for ns in sample_sizes:
    rejections = 0
    for _ in range(n_trials):
        x = rng.exponential(scale=true_scale, size=ns)
        _, scale_hat = stats.expon.fit(x, floc=0)
        _, p = stats.kstest(x, stats.expon(scale=scale_hat).cdf)
        if p < 0.05:
            rejections += 1
    print(f"  n={ns:5d}: rejection rate = {rejections/n_trials:.3f}")

---
## Try It Yourself

1. **Parametric bootstrap KS**: The standard KS p-value is biased when parameters are estimated from the data. Implement a parametric bootstrap: (a) fit the distribution, (b) simulate 1,000 samples of size n from the fitted distribution, (c) fit each simulated sample and compute its KS statistic against the re-fitted CDF, (d) compare your observed D_n to this bootstrap distribution. Does the corrected p-value change your decision?

2. **Anderson-Darling test**: The `scipy.stats.anderson` function implements the Anderson-Darling test, which weights the tails more heavily than KS. Apply it to the clinic interarrival data for the Exponential and Lognormal candidates. Does it give a different answer than KS?

3. **Wrong bins in χ²**: Repeat the chi-squared test using equal-width bins instead of equal-probability bins. What happens to bins in the upper tail? How does this affect the χ² statistic and the test decision?